# Limpeza e Pré-processamento

In [1]:
import html
import re
import unicodedata

import pandas as pd
import nltk
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from nltk.stem import RSLPStemmer, PorterStemmer
import spacy

pd.set_option("display.max_colwidth", 100)

df = pd.read_csv("../data/processed/reviews_raw.csv")
df = df.rename(columns={"review": "raw_text"})
print(f"Reviews carregadas: {len(df)}")
print(df["language"].value_counts())
df[["appid", "game_name", "language", "voted_up", "raw_text"]].head(3)


Reviews carregadas: 2000
language
english      1705
brazilian     295
Name: count, dtype: int64


,appid,game_name,language,voted_up,raw_text
0,730,Counter-Strike 2,english,True,would be pro if it werent laggy asf. stupid jitter ms
1,730,Counter-Strike 2,english,True,w game im gay and from israel
2,730,Counter-Strike 2,english,True,this is valoront for big boys


## Diagnóstico da amostra

Antes de escolher uma técnica, inspecionamos os dados para identificar problemas reais.


In [2]:
raw = df["raw_text"].fillna("")

is_empty = raw.str.strip() == ""
has_bbcode = raw.str.contains(r"\[/?[a-zA-Z\*][^\]]*\]", regex=True)
has_html_entity = raw.apply(lambda t: html.unescape(t) != t)
has_steam_heart_censor = raw.str.contains("♥", regex=False)
is_exact_duplicate = raw.duplicated(keep=False) & (~is_empty)
word_count_raw = raw.str.split().str.len().fillna(0)

print(f"Reviews vazias / só espaço:              {is_empty.sum()}")
print(f"Reviews com marcação BBCode (Steam):      {has_bbcode.sum()}")
print(f"Reviews com HTML entity (&amp; etc.):      {has_html_entity.sum()}")
print(f'Reviews com censura "♥♥♥" da Steam:        {has_steam_heart_censor.sum()}')
print(f"Reviews com texto EXATAMENTE duplicado:    {is_exact_duplicate.sum()}")
print(f"Idiomas na amostra:                        {sorted(df['language'].unique())}")
print()
print("Distribuição de nº de palavras (raw):")
print(word_count_raw.describe().round(1))


Reviews vazias / só espaço:              9
Reviews com marcação BBCode (Steam):      11
Reviews com HTML entity (&amp; etc.):      0
Reviews com censura "♥♥♥" da Steam:        55
Reviews com texto EXATAMENTE duplicado:    232
Idiomas na amostra:                        ['brazilian', 'english']

Distribuição de nº de palavras (raw):
count    2000.0
mean       18.9
std        57.7
min         0.0
25%         2.0
50%         5.0
75%        15.0
max      1206.0
Name: raw_text, dtype: float64


**O que a inspeção revelou:**

- Reviews com texto exatamente igual a outra review (memes/copypasta como `"good"`,
  `"nice"`, `"top"`, frases curtas repetidas por vários usuários).
- Boa parte das reviews é curta, confirmando a
  dificuldade já prevista na proposta ("reviews muito curtas oferecem pouco
  contexto").


## Limpeza

Cada transformação abaixo é justificada por um ganho concreto observado acima. Nada aqui apaga sentimento, negação ou ênfase
por isso pontuação e emojis não são removidos nesta etapa (ficam para o pré-processamento).


In [3]:
BBCODE_PATTERN = re.compile(r"\[/?[a-zA-Z\*][^\]]*\]")
EMOJI_PATTERN = re.compile(
    "[\U0001F300-\U0001FAFF\U00002600-\U000027BF\U0001F1E6-\U0001F1FF]"
)


def clean_text(raw_text: str) -> str:
    """Limpeza não destrutiva: decodifica entidades HTML, remove BBCode da
    Steam, normaliza Unicode (NFKC) e colapsa espaços/quebras de linha.
    Preserva pontuação, caixa e emojis, que ainda carregam sentimento/ênfase."""
    if pd.isna(raw_text):
        return ""
    text = html.unescape(raw_text)
    text = BBCODE_PATTERN.sub(" ", text)
    text = unicodedata.normalize("NFKC", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text


df["clean_text"] = df["raw_text"].apply(clean_text)

df["char_len"] = df["clean_text"].str.len()
df["word_count"] = df["clean_text"].str.split().str.len().fillna(0).astype(int)
df["is_empty"] = df["clean_text"].str.strip() == ""
df["is_too_short"] = df["word_count"] < 3
df["is_duplicate_text"] = df["clean_text"].duplicated(keep=False) & (~df["is_empty"])
df["has_emoji"] = df["clean_text"].apply(lambda t: bool(EMOJI_PATTERN.search(t)))
df["has_steam_heart_censor"] = df["clean_text"].str.contains("♥", regex=False)

print("Flags após a limpeza:")
print(df[["is_empty", "is_too_short", "is_duplicate_text", "has_emoji", "has_steam_heart_censor"]].sum())


Flags após a limpeza:
is_empty                    9
is_too_short              673
is_duplicate_text         268
has_emoji                  91
has_steam_heart_censor     55
dtype: int64


**Decisão sobre duplicatas:** não removemos automaticamente. Marcamos
`is_duplicate_text` e deixamos a decisão para cada tarefa: um classificador de
sentimento pode se beneficiar de removê-las (evita que uma frase repetida dezenas de
vezes domine o treino); já uma análise de frequência de opiniões pode querer
mantê-las. O dataset final preserva a flag para essa escolha ser feita depois, de
forma explícita.


### Tokenização

Usei o tokenizador do NLTK (Punkt + Treebank), que separa por espaço e isola pontuação.


In [4]:
def tokenize(text: str) -> list[str]:
    if not text:
        return []
    return word_tokenize(text)

df["tokens"] = df["clean_text"].apply(tokenize)
df["token_count"] = df["tokens"].apply(len)

df[["language", "clean_text", "tokens"]].head(3)


,language,clean_text,tokens
0,english,would be pro if it werent laggy asf. stupid jitter ms,"[would, be, pro, if, it, werent, laggy, asf, ., stupid, jitter, ms]"
1,english,w game im gay and from israel,"[w, game, im, gay, and, from, israel]"
2,english,this is valoront for big boys,"[this, is, valoront, for, big, boys]"


### Normalização, remoção de pontuação e stopwords

Para as tarefas que dependem de representações lexicais (TF-IDF, contagem de termos),
gerado uma segunda versão dos tokens: minúsculas e sem pontuação.

Stopwords são removidas por idioma.


In [5]:
STEAM_LANG_TO_NLTK = {"english": "english", "brazilian": "portuguese"}
STOPWORDS_BY_LANG = {
    steam_lang: set(stopwords.words(nltk_lang))
    for steam_lang, nltk_lang in STEAM_LANG_TO_NLTK.items()
}
print(f"Idiomas com stopwords disponíveis: {sorted(STOPWORDS_BY_LANG)}")

PUNCT_PATTERN = re.compile(r"^[\W_]+$", re.UNICODE)

def normalize_tokens(tokens: list[str]) -> list[str]:
    """Minúsculas + remoção de tokens puramente pontuação/símbolo."""
    return [t.lower() for t in tokens if not PUNCT_PATTERN.match(t)]

def remove_stopwords(tokens_norm: list[str], language: str) -> list[str]:
    stop_set = STOPWORDS_BY_LANG[language]
    return [t for t in tokens_norm if t not in stop_set]

df["tokens_norm"] = df["tokens"].apply(normalize_tokens)
df["tokens_no_stopwords"] = df.apply(
    lambda r: remove_stopwords(r["tokens_norm"], r["language"]), axis=1
)

df[["language", "tokens", "tokens_norm", "tokens_no_stopwords"]].head(3)


Idiomas com stopwords disponíveis: ['brazilian', 'english']


,language,tokens,tokens_norm,tokens_no_stopwords
0,english,"[would, be, pro, if, it, werent, laggy, asf, ., stupid, jitter, ms]","[would, be, pro, if, it, werent, laggy, asf, stupid, jitter, ms]","[would, pro, werent, laggy, asf, stupid, jitter, ms]"
1,english,"[w, game, im, gay, and, from, israel]","[w, game, im, gay, and, from, israel]","[w, game, im, gay, israel]"
2,english,"[this, is, valoront, for, big, boys]","[this, is, valoront, for, big, boys]","[valoront, big, boys]"


### stemming e lematização

Como a amostra agora tem só pt-br e inglês, aplicamos stemmer e lematizador específicos para cada idioma.

- **Português (`brazilian`)**: stemmer RSLP (NLTK) + lematizador spaCy `pt_core_news_sm`.
- **Inglês (`english`)**: stemmer Porter (NLTK) + lematizador spaCy `en_core_web_sm`.


In [6]:
rslp = RSLPStemmer()
porter = PorterStemmer()

def stem_tokens(tokens: list[str], language: str) -> list[str]:
    stemmer = rslp if language == "brazilian" else porter
    return [stemmer.stem(t) for t in tokens]

df["stemmed_tokens"] = df.apply(
    lambda r: stem_tokens(r["tokens_no_stopwords"], r["language"]), axis=1
)

pt_mask = df["language"] == "brazilian"
en_mask = df["language"] == "english"
print(f"pt-br: {pt_mask.sum()} reviews | inglês: {en_mask.sum()} reviews")

nlp_pt = spacy.load("pt_core_news_sm", disable=["parser", "ner"])
nlp_en = spacy.load("en_core_web_sm", disable=["parser", "ner"])

df["lemmas"] = pd.Series([None] * len(df), dtype=object)

pt_docs = nlp_pt.pipe(df.loc[pt_mask, "clean_text"].tolist())
df.loc[pt_mask, "lemmas"] = pd.Series(
    [[tok.lemma_.lower() for tok in doc if not tok.is_punct and not tok.is_space] for doc in pt_docs],
    index=df.loc[pt_mask].index, dtype=object,
)

en_docs = nlp_en.pipe(df.loc[en_mask, "clean_text"].tolist())
df.loc[en_mask, "lemmas"] = pd.Series(
    [[tok.lemma_.lower() for tok in doc if not tok.is_punct and not tok.is_space] for doc in en_docs],
    index=df.loc[en_mask].index, dtype=object,
)

df[["language", "tokens_no_stopwords", "stemmed_tokens", "lemmas"]].head(3)


pt-br: 295 reviews | inglês: 1705 reviews


,language,tokens_no_stopwords,stemmed_tokens,lemmas
0,english,"[would, pro, werent, laggy, asf, stupid, jitter, ms]","[would, pro, werent, laggi, asf, stupid, jitter, ms]","[would, be, pro, if, it, be, not, laggy, asf, stupid, jitter, ms]"
1,english,"[w, game, im, gay, israel]","[w, game, im, gay, israel]","[w, game, i, m, gay, and, from, israel]"
2,english,"[valoront, big, boys]","[valoront, big, boy]","[this, be, valoront, for, big, boy]"


In [7]:
sample_pt = df.loc[pt_mask, ["clean_text", "tokens_no_stopwords", "stemmed_tokens", "lemmas"]].head(3)
sample_en = df.loc[en_mask, ["clean_text", "tokens_no_stopwords", "stemmed_tokens", "lemmas"]].head(3)
print("Exemplos em português:")
display(sample_pt)
print("\nExemplos em inglês:")
display(sample_en)


Exemplos em português:


,clean_text,tokens_no_stopwords,stemmed_tokens,lemmas
3,passa raiva mas e bom kkkkkk,"[passa, raiva, bom, kkkkkk]","[pass, raiv, bom, kkkkkk]","[passar, raivo, mas, e, bom, kkkkkk]"
8,Bem legal,"[bem, legal]","[bem, legal]","[bem, legal]"
11,Bom jogo!,"[bom, jogo]","[bom, jog]","[bom, jogo]"



Exemplos em inglês:


,clean_text,tokens_no_stopwords,stemmed_tokens,lemmas
0,would be pro if it werent laggy asf. stupid jitter ms,"[would, pro, werent, laggy, asf, stupid, jitter, ms]","[would, pro, werent, laggi, asf, stupid, jitter, ms]","[would, be, pro, if, it, be, not, laggy, asf, stupid, jitter, ms]"
1,w game im gay and from israel,"[w, game, im, gay, israel]","[w, game, im, gay, israel]","[w, game, i, m, gay, and, from, israel]"
2,this is valoront for big boys,"[valoront, big, boys]","[valoront, big, boy]","[this, be, valoront, for, big, boy]"


## Comparação antes → depois


In [8]:
def vocab_size(token_lists):
    vocab = set()
    for toks in token_lists.dropna():
        vocab.update(toks)
    return len(vocab)

print("Tamanho do vocabulário em cada estágio:")
print(f"  tokens (brutos)          : {vocab_size(df['tokens'])}")
print(f"  tokens_norm (sem pontuação, minúsculo): {vocab_size(df['tokens_norm'])}")
print(f"  tokens_no_stopwords      : {vocab_size(df['tokens_no_stopwords'])}")
print(f"  stemmed_tokens (pt+en)   : {vocab_size(df['stemmed_tokens'])}")
print(f"  lemmas (pt+en)           : {vocab_size(df['lemmas'])}")

print()
print("Médias de nº de tokens por review:")
print(f"  raw (split por espaço)   : {word_count_raw.mean():.1f}")
print(f"  tokens_norm              : {df['tokens_norm'].apply(len).mean():.1f}")
print(f"  tokens_no_stopwords      : {df['tokens_no_stopwords'].apply(len).mean():.1f}")


Tamanho do vocabulário em cada estágio:
  tokens (brutos)          : 7225
  tokens_norm (sem pontuação, minúsculo): 6179
  tokens_no_stopwords      : 5974
  stemmed_tokens (pt+en)   : 4576
  lemmas (pt+en)           : 5250

Médias de nº de tokens por review:
  raw (split por espaço)   : 18.9
  tokens_norm              : 19.0
  tokens_no_stopwords      : 10.8


## Salvar dataset processado

In [9]:
def join_tokens(tokens):
    if not isinstance(tokens, list):
        return ""
    return " ".join(tokens)

output_columns = [
    "appid", "game_name", "recommendationid", "language", "voted_up",
    "timestamp_created", "author_playtime_at_review_hours",
    "raw_text", "clean_text",
    "char_len", "word_count", "token_count",
    "is_empty", "is_too_short", "is_duplicate_text",
    "has_emoji", "has_steam_heart_censor",
]

out = df[output_columns].copy()
out["tokens"] = df["tokens"].apply(join_tokens)
out["tokens_no_stopwords"] = df["tokens_no_stopwords"].apply(join_tokens)
out["stemmed_tokens"] = df["stemmed_tokens"].apply(join_tokens)
out["lemmas"] = df["lemmas"].apply(join_tokens)

out.to_csv("../data/processed/reviews_clean.csv", index=False)
print(f"Salvo: data/processed/reviews_clean.csv ({out.shape[0]} linhas, {out.shape[1]} colunas)")


Salvo: data/processed/reviews_clean.csv (2000 linhas, 21 colunas)
